# 02 — Data Cleaning

Loads the raw panel, handles types/missing values, derives log transforms, and writes the cleaned panel to `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv('../data/raw/panel_raw.csv')
raw.shape

## Basic cleaning

- Coerce numeric columns
- Drop rows with no economy identifier
- Cast `banking_crisis_dummy` to a clean 0/1 integer (NaN where undefined)

In [ ]:
df = raw.copy()

numeric_cols = ['syndicated_loans_gdp', 'corporate_bonds_gdp', 'banking_crisis_dummy',
                 'gdp_current_usd', 'gdp_growth_pct', 'inflation_pct']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna(subset=['economy'])
df['year'] = df['year'].astype(int)
df.shape

## Missingness overview

In [ ]:
df[numeric_cols].isna().mean().sort_values(ascending=False).to_frame('pct_missing')

## Log transforms for skewed ratios

Both financing ratios are typically right-skewed with a floor at 0. We add a small constant before logging to keep zeros usable.

In [ ]:
for col in ['syndicated_loans_gdp', 'corporate_bonds_gdp']:
    df[f'log_{col}'] = np.log1p(df[col])

## Save processed panel

In [ ]:
df.to_csv('../data/processed/panel_clean.csv', index=False)
print(f'Saved cleaned panel: {df.shape[0]} rows, {df.shape[1]} cols')